In [2]:
import json
import random
from pathlib import Path
from typing import Dict, List, Any

# Paths
BASE_DIR = Path("../data")
INPUT_FILE = BASE_DIR / "final_quiz_data.json"
OUTPUT_FILE = BASE_DIR / "backend_quiz_data.json"

print("✓ Setup complete")

✓ Setup complete


## Load Quiz Data

In [3]:
# Load the final quiz data
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    quiz_data = json.load(f)

print(f"✓ Loaded {len(quiz_data)} tests")

# Show structure
sample_test = quiz_data["test-01"]
print(f"\nSample test structure:")
print(f"  - Test ID: {sample_test['test_id']}")
print(f"  - Road Signs: {len(sample_test['sections']['road_signs'])}")
print(f"  - Priorities: {len(sample_test['sections']['priorities'])}")
print(f"  - Questions: {len(sample_test['sections']['general_questions'])}")

✓ Loaded 20 tests

Sample test structure:
  - Test ID: test-01
  - Road Signs: 16
  - Priorities: 8
  - Questions: 6


## Generate Wrong Answer Choices

For each question, we need to generate 3 wrong answers (distractors) to create multiple-choice questions.

**Strategy:**
- **Road Signs**: Use other sign descriptions from the same test or other tests
- **Priorities**: Use other priority descriptions
- **General Questions**: Use answers from other questions in the same test

In [4]:
def collect_all_answers_by_type(quiz_data: dict) -> dict:
    """Collect all answers grouped by question type."""
    all_answers = {
        "road_signs": {"ar": [], "fr": []},
        "priorities": {"ar": [], "fr": []},
        "general_questions": {"ar": [], "fr": []}
    }
    
    for test_name, test_data in quiz_data.items():
        sections = test_data["sections"]
        
        # Collect road sign answers
        for sign in sections["road_signs"]:
            if sign.get("answer_ar"):
                all_answers["road_signs"]["ar"].append(sign["answer_ar"])
            if sign.get("answer_fr") and sign["answer_fr"] != "###":
                all_answers["road_signs"]["fr"].append(sign["answer_fr"])
        
        # Collect priority answers
        for priority in sections["priorities"]:
            if priority.get("answer_ar"):
                all_answers["priorities"]["ar"].append(priority["answer_ar"])
            if priority.get("answer_fr") and priority["answer_fr"] != "###":
                all_answers["priorities"]["fr"].append(priority["answer_fr"])
        
        # Collect general question answers
        for question in sections["general_questions"]:
            if question.get("answer_ar"):
                all_answers["general_questions"]["ar"].append(question["answer_ar"])
            if question.get("answer_fr") and question["answer_fr"] != "###":
                all_answers["general_questions"]["fr"].append(question["answer_fr"])
    
    # Remove duplicates
    for category in all_answers:
        all_answers[category]["ar"] = list(set(all_answers[category]["ar"]))
        all_answers[category]["fr"] = list(set(all_answers[category]["fr"]))
    
    return all_answers

# Collect all answers
all_answers_pool = collect_all_answers_by_type(quiz_data)

print("Answer pools created:")
for answer_type, langs in all_answers_pool.items():
    print(f"  {answer_type}:")
    print(f"    Arabic: {len(langs['ar'])} unique answers")
    print(f"    French: {len(langs['fr'])} unique answers")

Answer pools created:
  road_signs:
    Arabic: 263 unique answers
    French: 0 unique answers
  priorities:
    Arabic: 144 unique answers
    French: 0 unique answers
  general_questions:
    Arabic: 80 unique answers
    French: 0 unique answers


In [5]:
def generate_wrong_choices(correct_answer_ar: str, correct_answer_fr: str, 
                          answer_pool_ar: list, answer_pool_fr: list, count: int = 3) -> list:
    """Generate wrong answer choices (bilingual) from the answer pools."""
    # Filter out the correct answer
    available_ar = [ans for ans in answer_pool_ar if ans != correct_answer_ar]
    available_fr = [ans for ans in answer_pool_fr if ans != correct_answer_fr]
    
    # Ensure we have enough wrong answers
    if len(available_ar) < count:
        print(f"⚠ Warning: Only {len(available_ar)} Arabic wrong answers available, need {count}")
        count = len(available_ar)
    
    # For French, if we don't have enough translated answers, use "###" placeholder
    if len(available_fr) < count:
        available_fr = available_fr + ["###"] * (count - len(available_fr))
    
    # Randomly select wrong answers (keep Arabic and French aligned by index)
    indices = random.sample(range(len(available_ar)), min(count, len(available_ar)))
    
    wrong_answers = []
    for idx in indices:
        wrong_ar = available_ar[idx]
        # Try to find matching French answer, otherwise use placeholder
        wrong_fr = available_fr[idx] if idx < len(available_fr) else "###"
        wrong_answers.append({"ar": wrong_ar, "fr": wrong_fr})
    
    return wrong_answers

# Test the function
test_correct_ar = "1- حذار، خطر غير معين."
test_correct_fr = "###"
test_wrong = generate_wrong_choices(
    test_correct_ar, test_correct_fr,
    all_answers_pool["road_signs"]["ar"], 
    all_answers_pool["road_signs"]["fr"], 
    3
)

print("Test wrong choices generation:")
print(f"  Correct AR: {test_correct_ar}")
print(f"  Correct FR: {test_correct_fr}")
print(f"  Wrong choices:")
for i, wrong in enumerate(test_wrong, 1):
    print(f"    {i}. AR: {wrong['ar'][:60]}...")
    print(f"       FR: {wrong['fr'][:60] if wrong['fr'] != '###' else '###'}...")

Test wrong choices generation:
  Correct AR: 1- حذار، خطر غير معين.
  Correct FR: ###
  Wrong choices:
    1. AR: 12- طريق وطني رقم 89....
       FR: ###...
    2. AR: 10- الدخول إلى منطقة التوقف فيها ممنوع....
       FR: ###...
    3. AR: 3- طريق ذات أولوية....
       FR: ###...


## Transform to Backend Format

Transform each test into the backend's expected structure:
- Each question becomes a `Question` object with choices
- Questions are categorized by type (SIGN, PRIORITY, RULE)
- Each choice has text, is_correct, and position

In [6]:
def transform_test_to_backend_format(test_data: dict, all_answers: dict) -> dict:
    """Transform a single test into backend-compatible format."""
    
    questions = []
    question_position = 1
    
    # 1. Process Road Signs
    for sign in test_data["sections"]["road_signs"]:
        answer_ar = sign.get("answer_ar", "")
        answer_fr = sign.get("answer_fr", "###")
        
        if not answer_ar:
            continue
        
        # Generate wrong choices
        wrong_choices = generate_wrong_choices(
            answer_ar, answer_fr,
            all_answers["road_signs"]["ar"], 
            all_answers["road_signs"]["fr"], 
            3
        )
        
        # Combine and create choices
        all_choices = [
            {"text_ar": answer_ar, "text_fr": answer_fr, "is_correct": True},
            {"text_ar": wrong_choices[0]["ar"], "text_fr": wrong_choices[0]["fr"], "is_correct": False},
            {"text_ar": wrong_choices[1]["ar"], "text_fr": wrong_choices[1]["fr"], "is_correct": False},
            {"text_ar": wrong_choices[2]["ar"], "text_fr": wrong_choices[2]["fr"], "is_correct": False},
        ]
        random.shuffle(all_choices)
        
        # Assign positions
        for i, choice in enumerate(all_choices):
            choice["position"] = i
        
        question_obj = {
            "text_ar": f"ما هي دلالة هذه الإشارة؟",
            "text_fr": f"Quelle est la signification de ce panneau ?",
            "image_url": sign["image_path"],
            "category": "SIGN",
            "type": "SINGLE_CHOICE",
            "difficulty": "MEDIUM",
            "is_required": True,
            "score": 1,
            "explanation": None,
            "tags": {
                "test_id": test_data["test_id"],
                "sign_number": sign["sign_number"],
                "section": "road_signs"
            },
            "choices": all_choices,
            "position": question_position
        }
        
        questions.append(question_obj)
        question_position += 1
    
    # 2. Process Priorities
    for priority in test_data["sections"]["priorities"]:
        answer_ar = priority.get("answer_ar", "")
        answer_fr = priority.get("answer_fr", "###")
        
        if not answer_ar:
            continue
        
        # Generate wrong choices
        wrong_choices = generate_wrong_choices(
            answer_ar, answer_fr,
            all_answers["priorities"]["ar"], 
            all_answers["priorities"]["fr"], 
            3
        )
        
        # Combine and create choices
        all_choices = [
            {"text_ar": answer_ar, "text_fr": answer_fr, "is_correct": True},
            {"text_ar": wrong_choices[0]["ar"], "text_fr": wrong_choices[0]["fr"], "is_correct": False},
            {"text_ar": wrong_choices[1]["ar"], "text_fr": wrong_choices[1]["fr"], "is_correct": False},
            {"text_ar": wrong_choices[2]["ar"], "text_fr": wrong_choices[2]["fr"], "is_correct": False},
        ]
        random.shuffle(all_choices)
        
        # Assign positions
        for i, choice in enumerate(all_choices):
            choice["position"] = i
        
        question_obj = {
            "text_ar": f"ما هو الترتيب الصحيح لمرور المركبات؟",
            "text_fr": f"Quel est l'ordre de passage correct ?",
            "image_url": priority["image_path"],
            "category": "PRIORITY",
            "type": "SINGLE_CHOICE",
            "difficulty": "MEDIUM",
            "is_required": True,
            "score": 1,
            "explanation": None,
            "tags": {
                "test_id": test_data["test_id"],
                "priority_number": priority["priority_number"],
                "section": "priorities"
            },
            "choices": all_choices,
            "position": question_position
        }
        
        questions.append(question_obj)
        question_position += 1
    
    # 3. Process General Questions
    for question in test_data["sections"]["general_questions"]:
        answer_ar = question.get("answer_ar", "")
        answer_fr = question.get("answer_fr", "###")
        
        if not answer_ar:
            continue
        
        # Generate wrong choices
        wrong_choices = generate_wrong_choices(
            answer_ar, answer_fr,
            all_answers["general_questions"]["ar"], 
            all_answers["general_questions"]["fr"], 
            3
        )
        
        # Combine and create choices
        all_choices = [
            {"text_ar": answer_ar, "text_fr": answer_fr, "is_correct": True},
            {"text_ar": wrong_choices[0]["ar"], "text_fr": wrong_choices[0]["fr"], "is_correct": False},
            {"text_ar": wrong_choices[1]["ar"], "text_fr": wrong_choices[1]["fr"], "is_correct": False},
            {"text_ar": wrong_choices[2]["ar"], "text_fr": wrong_choices[2]["fr"], "is_correct": False},
        ]
        random.shuffle(all_choices)
        
        # Assign positions
        for i, choice in enumerate(all_choices):
            choice["position"] = i
        
        question_obj = {
            "text_ar": question.get("question_ar", ""),
            "text_fr": question.get("question_fr", ""),
            "image_url": None,
            "category": "RULE",
            "type": "SINGLE_CHOICE",
            "difficulty": "MEDIUM",
            "is_required": True,
            "score": 1,
            "explanation": None,
            "tags": {
                "test_id": test_data["test_id"],
                "question_number": question["question_number"],
                "section": "general_questions"
            },
            "choices": all_choices,
            "position": question_position
        }
        
        questions.append(question_obj)
        question_position += 1
    
    return {
        "test_id": test_data["test_id"],
        "test_number": test_data["test_number"],
        "title": f"Test de Code {test_data['test_number']:02d}",
        "title_ar": f"اختبار رقم {test_data['test_number']:02d}",
        "description": f"Official driving test #{test_data['test_number']} with road signs, priorities, and general questions",
        "difficulty": "MEDIUM",
        "default_duration_sec": 1800,  # 30 minutes
        "settings": {
            "question_count": len(questions),
            "passing_score": int(len(questions) * 0.7),  # 70% to pass
            "randomize_questions": False,
            "randomize_choices": True
        },
        "is_public": True,
        "questions": questions
    }

# Test transformation on first test
print("Transforming test-01...")
backend_test_01 = transform_test_to_backend_format(
    quiz_data["test-01"], 
    all_answers_pool
)

print(f"✓ Transformed test-01:")
print(f"  Title: {backend_test_01['title']}")
print(f"  Questions: {len(backend_test_01['questions'])}")
print(f"  Passing score: {backend_test_01['settings']['passing_score']}/{backend_test_01['settings']['question_count']}")

Transforming test-01...
✓ Transformed test-01:
  Title: Test de Code 01
  Questions: 30
  Passing score: 21/30


## Transform All Tests

In [7]:
# Transform all tests
backend_data = {
    "tests": [],
    "metadata": {
        "total_tests": len(quiz_data),
        "generated_at": "2025-12-11",
        "version": "1.0",
        "format": "backend_compatible"
    }
}

for test_name, test_data in quiz_data.items():
    print(f"Processing {test_name}...", end=" ")
    
    backend_test = transform_test_to_backend_format(test_data, all_answers_pool)
    backend_data["tests"].append(backend_test)
    
    print(f"✓ {len(backend_test['questions'])} questions")

print(f"\n{'='*60}")
print(f"Transformation complete!")
print(f"  Total tests: {len(backend_data['tests'])}")
print(f"  Total questions: {sum(len(test['questions']) for test in backend_data['tests'])}")
print(f"{'='*60}")

Processing test-01... ✓ 30 questions
Processing test-02... ✓ 30 questions
Processing test-03... ✓ 30 questions
Processing test-04... ✓ 30 questions
Processing test-05... ✓ 30 questions
Processing test-06... ✓ 30 questions
Processing test-07... ✓ 30 questions
Processing test-08... ✓ 29 questions
Processing test-09... ✓ 30 questions
Processing test-10... ✓ 30 questions
Processing test-11... ✓ 30 questions
Processing test-12... ✓ 30 questions
Processing test-13... ✓ 30 questions
Processing test-14... ✓ 30 questions
Processing test-15... ✓ 30 questions
Processing test-16... ✓ 30 questions
Processing test-17... ✓ 30 questions
Processing test-18... ✓ 30 questions
Processing test-19... ✓ 30 questions
Processing test-20... ✓ 30 questions

Transformation complete!
  Total tests: 20
  Total questions: 599


## Save Backend Data

In [8]:
# Save to file
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(backend_data, f, ensure_ascii=False, indent=2)

print(f"✓ Saved to: {OUTPUT_FILE}")
print(f"\nFile size: {OUTPUT_FILE.stat().st_size / 1024 / 1024:.2f} MB")

✓ Saved to: ..\data\backend_quiz_data.json

File size: 1.18 MB


## Verify Output Structure

In [9]:
# Display sample question from test-01
sample_test = backend_data["tests"][0]
sample_question = sample_test["questions"][0]

print("="*60)
print("SAMPLE QUESTION")
print("="*60)
print(f"Test: {sample_test['title']}")
print(f"Question #{sample_question['position']}")
print(f"Text AR: {sample_question['text_ar']}")
print(f"Text FR: {sample_question['text_fr']}")
print(f"Image: {sample_question['image_url']}")
print(f"Category: {sample_question['category']}")
print(f"Type: {sample_question['type']}")
print(f"\nChoices:")
for choice in sample_question['choices']:
    marker = "✓" if choice['is_correct'] else " "
    print(f"  [{marker}] Position {choice['position']}:")
    print(f"      AR: {choice['text_ar'][:60]}...")
    print(f"      FR: {choice['text_fr'][:60] if choice['text_fr'] != '###' else '###'}...")

print("\n" + "="*60)

SAMPLE QUESTION
Test: Test de Code 01
Question #1
Text AR: ما هي دلالة هذه الإشارة؟
Text FR: Quelle est la signification de ce panneau ?
Image: data/individual_signs/test-01/test-01_sign_01.png
Category: SIGN
Type: SINGLE_CHOICE

Choices:
  [ ] Position 0:
      AR: 9- الدروان إلى الخلف ممنوع....
      FR: ###...
  [✓] Position 1:
      AR: 1- حذار، خطر غير معين....
      FR: ###...
  [ ] Position 2:
      AR: 16- قف على بعد 150 متر....
      FR: ###...
  [ ] Position 3:
      AR: 9- حذار، منعرج إلى اليسار....
      FR: ###...



## Statistics & Verification

In [10]:
# Calculate statistics
stats = {
    "total_tests": len(backend_data["tests"]),
    "total_questions": 0,
    "questions_by_category": {"SIGN": 0, "PRIORITY": 0, "RULE": 0},
    "questions_by_test": [],
    "total_choices": 0,
    "correct_choices": 0
}

for test in backend_data["tests"]:
    test_question_count = len(test["questions"])
    stats["total_questions"] += test_question_count
    stats["questions_by_test"].append(test_question_count)
    
    for question in test["questions"]:
        stats["questions_by_category"][question["category"]] += 1
        stats["total_choices"] += len(question["choices"])
        stats["correct_choices"] += sum(1 for c in question["choices"] if c["is_correct"])

print("="*60)
print("BACKEND DATA STATISTICS")
print("="*60)
print(f"Total Tests: {stats['total_tests']}")
print(f"Total Questions: {stats['total_questions']}")
print(f"\nQuestions by Category:")
for category, count in stats["questions_by_category"].items():
    print(f"  {category}: {count}")
print(f"\nAverage Questions per Test: {stats['total_questions'] / stats['total_tests']:.1f}")
print(f"Min Questions in a Test: {min(stats['questions_by_test'])}")
print(f"Max Questions in a Test: {max(stats['questions_by_test'])}")
print(f"\nTotal Choices: {stats['total_choices']}")
print(f"Correct Choices: {stats['correct_choices']}")
print(f"Choices per Question: {stats['total_choices'] / stats['total_questions']:.1f}")
print("="*60)

# Verify each question has exactly 1 correct answer
print("\nVerifying answer correctness...")
issues = []
for test in backend_data["tests"]:
    for question in test["questions"]:
        correct_count = sum(1 for c in question["choices"] if c["is_correct"])
        if correct_count != 1:
            issues.append(f"{test['test_id']} - Question {question['position']}: {correct_count} correct answers")

if issues:
    print(f"⚠ Found {len(issues)} issues:")
    for issue in issues[:10]:  # Show first 10
        print(f"  - {issue}")
else:
    print("✓ All questions have exactly 1 correct answer")

BACKEND DATA STATISTICS
Total Tests: 20
Total Questions: 599

Questions by Category:
  SIGN: 320
  PRIORITY: 160
  RULE: 119

Average Questions per Test: 29.9
Min Questions in a Test: 29
Max Questions in a Test: 30

Total Choices: 2396
Correct Choices: 599
Choices per Question: 4.0

Verifying answer correctness...
✓ All questions have exactly 1 correct answer
